# 06_load_mongodb — Incremental Sink to MongoDB Atlas (new notebook)

Reads **only the newly inserted or updated rows** from `gold.articles`' 
Delta Change Data Feed and pushes them to MongoDB Atlas with a 
`bulk_write([UpdateOne(..., upsert=True), ...])` — the collection is never 
truncated, and a run that processes zero changed rows does nothing at all.

**Read side:** identical pattern to `04_gold_aggregation`: 
`spark.readStream.format("delta").option("readChangeFeed", 
"true").option("startingVersion", 0).table(TABLES["gold"])`, 
`foreachBatch`, `trigger(availableNow=True)`. `startingVersion=0` only 
matters on the very first run (no checkpoint yet); every run after that 
resumes from the checkpoint and only sees genuinely new changes.

**Write side:** `pymongo`'s synchronous driver, `bulk_write` with one 
`UpdateOne(filter={"Article_ID": ...}, update={"$set": doc}, 
upsert=True)` per changed row, batched per Spark micro-batch (not one 
round-trip per row).

**Connection:** `MONGO_URI` / `MONGO_DB_NAME` / `MONGO_COLLECTION_NAME` 
come from `00_config` — never hardcoded here, same widget-then-env-var 
pattern as the storage connection string.

In [0]:
%run ./00_config

# 00_config — Shared Config & Helpers

`%run` by every other notebook. Centralizes the storage connection string, 
source list, table/volume names, and small logging/counting helpers.

**This is the connection-string variant of the pipeline.** Bronze reads 
files landed in ADLS using the plain Python `azure-storage-blob` SDK and a 
storage account connection string — not Auto Loader over an `abfss://` 
path, and not a Unity Catalog External Location. That's a deliberate 
choice, not a step backward: Databricks Serverless blocks 
`spark.conf.set("fs.azure.account.key...")` outright (`CONFIG_NOT_AVAILABLE`), 
which is what Auto Loader over `abfss://` would need without an External 
Location, and the External Location route needs grants (`READ FILES`, 
`WRITE FILES`, `CREATE MANAGED STORAGE`) that may not be available to you 
on a shared/class storage account. The Blob SDK never touches Spark's 
filesystem config at all -- it's a plain HTTPS call, same mechanism the 
live-fetch version (`py_work/`) already uses successfully against public 
APIs on serverless -- so it works regardless of either restriction.

**Note on architecture:** Bronze/Silver/Gold are **Unity Catalog managed 
tables** — `fomoless.bronze.articles_raw`, `fomoless.silver.articles`, 
`fomoless.silver.articles_rejected`, `fomoless.gold.articles`. `01_setup` 
creates the schemas (assumes the `fomoless` catalog itself already exists — 
see the note in `01_setup` if it doesn't yet).

**Bronze is a plain batch step, not a stream** — since there's no Auto 
Loader checkpoint tracking "what's already been read", incrementality 
comes from checking each landed file's blob name against what's already in 
`bronze.articles_raw` (`source_blob_name`) before downloading it, so a 
re-run only pulls genuinely new files. **Silver stays incremental** via its 
own Structured Streaming checkpoint, stored in a small Unity Catalog 
**managed Volume** (`VOLUME_PATH`, created by `01_setup`) — Databricks-managed 
storage under the catalog you already own, so it needs no extra grants 
either. That Volume also holds nothing else; dedup-merge audit logs go 
straight to ADLS via the same Blob SDK connection as bronze.

---
**SECURITY FIX (this revision):** a previous version of this notebook had 
a live Azure Storage account key hardcoded as the `connection_string` 
widget's default value. That key has been removed from this file — **if 
you're seeing this note, rotate that key in Azure Portal (Storage Account 
→ Access keys → Rotate) if you haven't already**, since it was exposed in 
a shared notebook. Going forward, set the connection string via the widget 
at run time or the `AZURE_STORAGE_CONNECTION_STRING` environment variable 
— never as a hardcoded default.

**New in this revision:** `CANONICAL_SOURCE_NAMES` (CHK-4 enum gate), 
`SOURCES_WITHOUT_NATIVE_SUMMARY` (CHK-5 conditional-completeness exception 
list), and `CLOCK_SKEW_LOG_PREFIX` (CHK-7 audit trail) — all consumed by 
`03_silver_transform`.

---
**Gold/Mongo revision:** added `ensure_cdf_enabled` (moved here from
`03_silver_transform` since `04_gold_aggregation` now needs the identical
logic — shared, not duplicated), `TREND_DECAY_HALF_LIFE_DAYS` (TR-11) and
the `CROSS_SOURCE_*` / `ENGAGEMENT_*` / `MOMENTUM_MULTIPLIER_*` constants
(TR-13, revised — see below), new `gold`/`mongo_sink` checkpoint paths,
and `MONGO_URI` (never hardcoded).

**TR-13 REVISED (this pass):** the old flat `MOMENTUM_MULTIPLIERS` lookup
only ever had one confirmed value (`BBC`/`The Guardian` → `3.0`), with
every other source silently defaulting to `1.0` — which is exactly what
let BBC dominate the trend rankings regardless of whether any individual
BBC article was actually resonating. It's been replaced with two
data-driven signals computed in `04_gold_aggregation`'s
`with_momentum_signals`: how many *other* sources are covering something
with an overlapping tag (`cross_source_score`), and how this article's
engagement ranks against its *own* source's recent history
(`engagement_score`). Both are derived from data already in the pipeline —
no new external dependency, and no business input required to be
structurally complete. See that function's docstring for the full
reasoning; the constants below only tune the computation.

## 1. Serverless "spark not initialized" workaround

On some Databricks Serverless sessions, the notebook-provided `spark` 
global isn't ready the first time a cell references it, raising 
`SystemError: Internal error: spark should be initialized with the first 
notebook command.` Explicitly building a session via `databricks.connect` 
fixes it and is a no-op anywhere `spark` is already fine.

## 2. Storage connection string

From Azure Portal → your storage account → **Access keys** → Connection 
string (key 1 or key 2, either works). Settable via a Databricks widget 
(`connection_string`) or the `AZURE_STORAGE_CONNECTION_STRING` environment 
variable -- never hardcode it directly in a cell.

## 3. Sources & schema

## 4. Catalog, schemas, tables & volume

## 5. Gold trend-scoring constants (TR-11, TR-13)

## 6. MongoDB connection (for `06_load_mongodb`)

Same pattern as the storage connection string above: widget first, then 
environment variable, never a hardcoded default. `MONGO_URI` should 
already contain the target database name (e.g. 
`.../fomoless?retryWrites=true&w=majority`); `MONGO_COLLECTION_NAME` names 
the collection within it.

## 7. Helpers

[2026-09-26 14:00:10 UTC] Config loaded — container: fomoless, catalog: fomoless


In [0]:
import math
from datetime import datetime as _dt
import decimal
import numpy as np

from pyspark.sql import functions as F
from pymongo import MongoClient, UpdateOne

## 1. Mongo client

One client per notebook run (not per micro-batch) — `pymongo.MongoClient` 
is thread-/connection-pool-safe to reuse across `foreachBatch` calls, and 
creating a fresh one per batch would just add needless connection 
overhead.

In [0]:
if not MONGO_URI:
    raise EnvironmentError(
        "No MongoDB URI found. Supply it via the Databricks notebook widget "
        "'mongo_uri', or set MONGODB_ATLAS_URI as an environment variable."
    )

mongo_client = MongoClient(MONGO_URI)
mongo_collection = mongo_client[MONGO_DB_NAME][MONGO_COLLECTION_NAME]
log(f"Connected to MongoDB Atlas — db: {MONGO_DB_NAME}, collection: {MONGO_COLLECTION_NAME}")

[2026-09-26 14:00:11 UTC] Connected to MongoDB Atlas — db: fomoless, collection: articles


## 2. Row → Mongo document conversion

pandas reintroduces `NaN`/`NaT` for anything that round-tripped through 
`.toPandas()`, even where the source was a genuine SQL `NULL` — this 
normalizes those (and Spark's CDF bookkeeping columns) out before a 
document ever reaches Mongo.

In [0]:
CDF_METADATA_COLUMNS = {"_change_type", "_commit_version", "_commit_timestamp"}


def clean_mongo_value(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    if isinstance(value, decimal.Decimal):
        return float(value)
    if isinstance(value, np.ndarray):
        return [clean_mongo_value(v) for v in value.tolist()]
    if isinstance(value, np.generic):
        return value.item()
    return value


def row_to_mongo_doc(row: dict) -> dict:
    return {
        k: clean_mongo_value(v) for k, v in row.items()
        if k not in CDF_METADATA_COLUMNS
    }

## 3. Per-micro-batch processing

Same `_change_type` filter as `04_gold_aggregation`, for the same reason: 
`update_preimage` is the stale value being replaced, and `delete` is 
filtered defensively even though nothing in this pipeline currently 
issues one.

In [0]:
def process_mongo_batch(batch_df, batch_id: int) -> None:
    changed = batch_df.filter(F.col("_change_type").isin("insert", "update_postimage"))

    df_pandas = changed.toPandas()
    if df_pandas.empty:
        log(f"[mongo batch {batch_id}] No inserted/updated rows in this CDF batch — nothing to do.")
        return

    docs = [row_to_mongo_doc(row) for row in df_pandas.to_dict(orient="records")]

    operations = [
        UpdateOne({"Article_ID": doc["Article_ID"]}, {"$set": doc}, upsert=True)
        for doc in docs
        if doc.get("Article_ID") is not None
    ]
    skipped = len(docs) - len(operations)
    if skipped > 0:
        log(f"[mongo batch {batch_id}] ⚠️  Skipped {skipped} row(s) with a null Article_ID (can't upsert without a key)")

    if not operations:
        log(f"[mongo batch {batch_id}] Nothing upsertable in this batch.")
        return

    with MongoClient(MONGO_URI) as batch_client:
        batch_collection = batch_client[MONGO_DB_NAME][MONGO_COLLECTION_NAME]
        result = batch_collection.bulk_write(operations, ordered=False)

    log(
        f"[mongo batch {batch_id}] bulk_write: "
        f"{result.upserted_count} inserted, {result.modified_count} updated "
        f"(of {len(operations)} operation(s) sent)"
    )

## 4. Stream gold's change feed through it

In [0]:
query = (
    spark.readStream.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .table(TABLES["gold"])
    .writeStream
    .foreachBatch(process_mongo_batch)
    .option("checkpointLocation", CHECKPOINTS["mongo_sink"])
    .trigger(availableNow=True)
    .start()
)

run_stream(query, "mongo_sink")

In [0]:
mongo_count = mongo_collection.count_documents({})
log(f"MongoDB Atlas — {MONGO_DB_NAME}.{MONGO_COLLECTION_NAME} total documents: {mongo_count}")

[2026-09-26 14:00:54 UTC] MongoDB Atlas — fomoless.articles total documents: 3924
